# Diagnostics · Leave-one-basin-out CV

**Primary interface** for the honest, low-variance combined-model metrics. This
notebook calls `channel_heads.eval` — `leave_one_group_out_oof`,
`max_precision_threshold`, `classification_metrics` — the same primitives the
batch wrapper `scripts/eval_lobo_cv.py` uses (no duplicated CV/metric logic).

Each basin is held out in turn; the model trains on the rest and predicts the
held-out basin. We report pooled out-of-fold AUC, per-fold AUC mean±std, and the
threshold-decision metrics.

It is **read-only**: it displays the comparison table inline and writes no CSV.

> Trains many small XGBoost models (one per basin per config) — takes ~1–2 min.

In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from channel_heads.io.paths import PROJECT_ROOT
from channel_heads.eval import (
    classification_metrics,
    leave_one_group_out_oof,
    max_precision_threshold,
)

GEOM = ["orientation_diff_deg", "headhead_dist_norm", "apex_angle_deg",
        "strahler_order_diff", "proximity_profile_norm"]
EMB = [f"emb_{i}" for i in range(4)]
FEATURES = GEOM + EMB
N_ESTIMATORS, MAX_DEPTH, LR, SEED = 200, 4, 0.1, 42

DATASETS = {
    "baseline": PROJECT_ROOT / "data/results/master_dataset_v4_cnn_full.csv",
    "regA": PROJECT_ROOT / "data/results/master_dataset_regA_with_emb.csv",
    "regB": PROJECT_ROOT / "data/results/master_dataset_regB_with_emb.csv",
    "regC": PROJECT_ROOT / "data/results/master_dataset_regC_with_emb.csv",
}
available = {k: v for k, v in DATASETS.items() if v.exists()}
print("available configs:", list(available))

available configs: ['baseline', 'regA', 'regB', 'regC']


## Run LOBO per config — via the package

In [2]:
def fit_predict(X_tr, y_tr, X_te):
    # Per-fold geom+emb XGBoost (the model spec; CV mechanics live in the package)
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    model = XGBClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, learning_rate=LR,
        scale_pos_weight=spw, eval_metric="logloss", random_state=SEED,
    )
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_te)[:, 1]


rows = []
for name, path in available.items():
    df = pd.read_csv(path).dropna(subset=FEATURES + ["y", "basin"]).reset_index(drop=True)
    oof, y, groups, fold_aucs = leave_one_group_out_oof(df, FEATURES, fit_predict)
    thr = max_precision_threshold(y, oof)
    m = classification_metrics(y, oof, thr)
    rows.append({
        "config": name,
        "n": len(df),
        "n_basins": groups.nunique(),
        "pooled_auc": m["roc_auc"],
        "fold_auc_mean": float(np.mean(fold_aucs)),
        "fold_auc_std": float(np.std(fold_aucs)),
        "threshold": thr,
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "accuracy": m["accuracy"],
    })
    print(f"{name:9} pooled_AUC={m['roc_auc']:.3f}  "
          f"fold_AUC={np.mean(fold_aucs):.3f}±{np.std(fold_aucs):.3f}  F1={m['f1']:.3f}")

baseline  pooled_AUC=0.916  fold_AUC=0.888±0.092  F1=0.642
regA      pooled_AUC=0.887  fold_AUC=0.853±0.063  F1=0.607
regB      pooled_AUC=0.889  fold_AUC=0.870±0.081  F1=0.614
regC      pooled_AUC=0.710  fold_AUC=0.908±0.046  F1=0.452


## Comparison table (read-only — no CSV written)

In [3]:
pd.DataFrame(rows).set_index("config").round(4)

,n,n_basins,pooled_auc,fold_auc_mean,fold_auc_std,threshold,precision,recall,f1,accuracy
config,,,,,,,,,,
baseline,5413,17,0.9157,0.8876,0.0918,0.8687,0.8950,0.5009,0.6423,0.7755
regA,23742,17,0.8866,0.8531,0.0630,0.7694,0.7683,0.5011,0.6065,0.8063
regB,10612,17,0.8886,0.8703,0.0813,0.7830,0.7905,0.5024,0.6143,0.8018
regC,28954,17,0.7102,0.9084,0.0462,0.4444,0.4119,0.5004,0.4519,0.6913


---
Persist the full metrics table (writes `models/lobo_cv_metrics.csv`):

```bash
python scripts/eval_lobo_cv.py
```